In [1]:
import os
from circuit_tracer.subgraph.prune import prune_graph_pipeline, PruneGraph
from circuit_tracer.subgraph.api import generate_graph, get_feature, save_subgraph
from circuit_tracer.subgraph.classify import classify_features
# from circuit_tracer.subgraph.group_llm import grouping_pipeline
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.create_graph_files import create_graph_files  
from circuit_tracer.graph import Graph, prune_graph, compute_graph_scores
from transformers import AutoModelForCausalLM, AutoTokenizer
import shap

## Load LLM

In [2]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Use SHAP for token weights

In [4]:
explainer = shap.Explainer(model, tokenizer)

In [5]:
prompts = ['Fact: The capital of the state containing Dallas is']

In [ ]:
shap_values = explainer(prompts)

In [ ]:
shap.plots.text(shap_values)

In [52]:
print(shap_values)

.values =
array([[[-4.22446732e-06],
        [ 6.54286429e+00],
        [ 1.68508452e+00],
        [ 1.20230146e+00],
        [ 6.04736818e+00],
        [ 6.77137808e-02],
        [ 4.40673760e+00],
        [ 4.12085123e-01],
        [ 4.40843961e-01]]])

.base_values =
array([[-21.73993724]])

.data =
(array(['', 'Cat', ' is', ' to', ' kitten', ' as', ' dog', ' is', ' to'],
      dtype=object),)


In [34]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze()) # How to normalize Shap values? softmax, relu first then normalize, ...
print(token_weights)

[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
 0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]


## Generate graph if not exist

In [4]:
from generate_new_graphs import download_graph
download_graph(
    prompt=prompts[0],
    slug="austin_plt",
    source_set = 'gemmascope-transcoder-16k',
    out_path="temp_graph_files/austin_plt.json",
    node_threshold = 0.95,
    edge_threshold = 0.98,
)

requesting graph for slug='austin_plt' prompt='Fact: The capital of the state containing Dallas is...'


downloading graph json from https://neuronpedia-attrib.s3.us-east-1.amazonaws.com/user-graphs/anonymous/austin_plt.json
saved austin_plt -> temp_graph_files/austin_plt.json (nodes=5387)


## ASG Pipeline

### Prune

In [ ]:
save_prune_graph('subgraph/puppy_plt_shap_06_08.pt')

NameError: name 'save_prune_snapshot' is not defined

In [16]:
from circuit_tracer.subgraph.prune import prune_graph_pipeline
from circuit_tracer.subgraph.cluster import cluster_graph, cluster_graph_with_labels
from circuit_tracer.subgraph.auto_grouping import find_best_k

# prune_graph = load_prune_graph('subgraph/puppy_clt_shap_07_085.pt')
token_weights = [0,0,0,0,1/3,0,0,1/3,0,1/3,0]
# 1) Build prune_graph as usual
prune_graph = prune_graph_pipeline(
    json_path="temp_graph_files/austin_clt.json",
    token_weights = token_weights,
    logit_weights="target",
    node_threshold=0.5,
    edge_threshold=0.98,
    alpha=0.6,
    keep_all_tokens_and_logits=False,
    filter_act_density=False,
)


In [17]:
prune_graph.num_nodes, prune_graph.num_edges

(19, 106)

In [18]:
for node_id in prune_graph.kept_ids:
    print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_scores[prune_graph.kept_ids.index(node_id)].item())


1_12928_10  AssemblyCulture 0.01099186297506094
1_72774_10 Code/licensing snippets 0.011941816657781601
16_89970_9 Texas 0.014559511095285416
17_1822_10 place names 0.01160102803260088
18_3623_10 capital cities 0.014878886751830578
20_44686_9 Texas locations/legal contexts 0.02001420594751835
20_44686_10 Texas locations/legal contexts 0.02730736695230007
20_74108_10 capital 0.01564362272620201
21_16875_10 cities 0.011956778354942799
21_61721_10 code/documentation 0.013235933147370815
21_84338_10 Cities and locations 0.01867927797138691
22_11998_10 Texas and Dallas 0.021464185789227486
22_32893_10 Texas legal documents 0.016960259526968002
23_31673_10 Court appeals at specific location 0.010377970524132252
24_79427_10  Kara 0.01839389093220234
E_6037_4 Emb: "  capital" 0.18746185302734375
E_2329_7 Emb: "  state" 0.11353735625743866
E_26865_9 Emb: "  Dallas" 0.2893992066383362
27_22605_10 Output " Austin" (p=0.439) 0.6248607635498047


In [ ]:
# from circuit_tracer.subgraph.prune import save_prune_graph, load_prune_graph
# save_prune_graph(prune_graph, "subgraph/austin_plt_shap_07_07.pt")
# loaded_prune_graph = load_prune_graph("temp_graph_files/austin_clt_prune

In [8]:
# 2) Auto-select k
best_k, sweep = find_best_k(
    prune_graph,
    max_layer_span=4,
    alpha=0,
    beta=0,
    # optional knobs:
    # k_min_override=3,
    # k_max_override=12,
    # max_sn=10,
    # weights={"w_intra": 0.3, "w_dag": 0.25, "w_flow": 0.25, "w_size": 0.2},
)

print("best_k:", best_k)
print("best score:", sweep[best_k]["total"] if best_k in sweep else None)

# 3) Run final clustering with best_k
supernodes = cluster_graph_with_labels(
    prune_graph,
    target_k=best_k,
    max_layer_span=4,
)

best_k: 2
best score: 0.6574615846025443


In [45]:
supernodes

[['cluster_0', '6_36271_6', '7_62570_6'],
 ['cluster_1', '8_82678_4', '12_36166_4'],
 ['cluster_3',
  '18_32351_8',
  '18_13641_6',
  '18_46270_6',
  '18_13641_8',
  '17_86326_8'],
 ['cluster_4', '23_89288_8', '21_25015_8', '19_79175_8', '22_84930_8'],
 ['cluster_5', '19_6382_8', '20_15972_8', '20_83104_8', '21_40843_8']]

In [46]:
for supernode in supernodes:
    print(supernode[0])
    for node_id in supernode[1:]:
        print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_influence[prune_graph.kept_ids.index(node_id)].item(), prune_graph.node_relevance[prune_graph.kept_ids.index(node_id)].item())

cluster_0
6_36271_6 pets and veterinary care 0.6901833415031433 0.8496087789535522
7_62570_6 Animals and breeds 0.5998135805130005 0.8181249499320984
cluster_1
8_82678_4 puppy 0.6823090314865112 0.814883291721344
12_36166_4 children 0.680974006652832 0.6365798115730286
cluster_3
18_32351_8 baby animals 0.5582504868507385 0.5373765826225281
18_13641_6 dog 0.5675403475761414 0.550097644329071
18_46270_6 dog 0.6584874391555786 0.6315413117408752
18_13641_8 dog 0.5383545160293579 0.504685640335083
17_86326_8 baby animals 0.6599850654602051 0.6063694357872009
cluster_4
23_89288_8 Relating to dogs 0.5632526278495789 0.4652628004550934
21_25015_8 Vets and pets 0.6755582094192505 0.5425448417663574
19_79175_8  your 0.6508617401123047 0.6212460994720459
22_84930_8 animals 0.6023414731025696 0.5104546546936035
cluster_5
19_6382_8 Non-English vocabulary 0.6782881617546082 0.6398190855979919
20_15972_8 can 0.552528440952301 0.5525699853897095
20_83104_8 animals 0.5754494667053223 0.482928812503814

In [7]:
import json
from typing import List
def extract_supernodes(flow_map_path: str) -> List[List[str]]:
    """
    Load a mapping JSON like flow_analysis_supernode_map.json and return
    a list of supernodes in the format:
      [ ["SN_LABEL", "node_a", "node_b", ...], ... ]
    """
    p = Path(flow_map_path)
    with p.open("r", encoding="utf-8") as f:
        mapping = json.load(f)

    supernodes = [[label, *nodes] for label, nodes in mapping.items()]
    return supernodes

### Visualize on the web

In [23]:
import json
from circuit_tracer.subgraph.api import save_subgraph

# Expected format for save_subgraph:
# supernodes = [
#   ["label_1", "node_id_a", "node_id_b"],
#   ["label_2", "node_id_c", "node_id_d", "node_id_e"],
# ]
# supernodes = extract_supernodes("flow_analysis_supernode_map.json")
# print(supernodes)


In [41]:
status, body = save_subgraph(
    modelId="gemma-2-2b",
    slug="factthecapitalof-1771775020852",                       # parent graph slug
    displayName="austin-grouped-shap",     # subgraph name shown on Neuronpedia
    pinnedIds=prune_graph.kept_ids,                  # or PruneGraph.kept_ids
    supernodes=supernodes,               # output from grouping pipeline
    pruningThreshold=0.7,
    densityThreshold=0.99,
)

print("status:", status)
try:
    print(json.loads(body))
except Exception:
    print(body)

{'Content-Type': 'application/json', 'x-api-key': 'sk-np-Vi0OcQl8zNm9jC7nyGRYfwssvSakMyrPjV675uEWIuU0'}


status: 200
{'success': True, 'subgraphId': 'cmnybzrgv0002uz3gx86q78xb'}


# TODO
List of things that need more experiments/improvements in descending order of importance for each step
## Pruning
1. Choose threshold (currently just sweep the threshold until we have ~30 nodes)
2. Initialize with Shap (normalize) (softmax | relu + normalize | +base_value then normalize)
3. Normalize adjacency matrix (currently adjacency.abs() then normalize (negative edges still contribute to influence and relevance))

## Grouping

Objectives to group features:
- similar embeddings (in classic SAE usually decoder vectors)
- simliar edges/logit effects (promoting the same feature or the same logit)
- similar contexts (activating on similar tokens, concepts, contexts)
- based on the claim we make about the mechanism (we might want to preserve the paths)

Approaches tried:
- Greedy constraint grouping
    + built distance graph based on BERT embeddings of autointerp (+ feature type) and group with antichain constraint (don't group nodes with path to each other)
    + input: autointerp + adj matrix
    + problems: constraints too strict, sensitive on the edges of the graph; did not consider local role of the features.
- Prompt LLM: 
    + Classify feature type based on feature details
    + Group using feature types + autointerp
    + problems: cover context and global role of the features, but not local role (in this exact graph) -> might not be mechanisticly sensible
    
Approaches to try:
- Clustering by (decoder vectors or autointerp) + edge weights similarity, constraint by difference in layers (may be sweep diff_layer = 0, 1, ... and merge iteratively)
- Higher order graphs




## Eval
1. Test intervention api
2. Find dataset
3. Hypothesis testing: Mechanism Preservation, Mechanism Localization, Minimality